# Multi-view 3D Reconstruction via Direct Linear Triangulation (DLT)

In this practice, you will implement **3-view triangulation** to recover the 3D position of points from their 2D projections in three calibrated cameras (**left**, **mid**, **right**). You are given the projection matrices $P_l, P_m, P_r$ and 7 corresponding 2D points across the three views.

By the end of this practice, you will understand:
- How a 2D image point and a 3×4 projection matrix together give a linear constraint on the 3D point — the **DLT formulation**
- Why we use the **cross-product (skew-symmetric) matrix** $[\mathbf{x}]_\times$ to convert $\lambda \mathbf{x} = P\mathbf{X}$ into a homogeneous linear system $A\mathbf{X} = \mathbf{0}$
- How to combine constraints from multiple views and solve $A\mathbf{X} = \mathbf{0}$ with **SVD**
- How to verify a reconstruction by **reprojection** back into each camera

### Due date
2026/5/19  23:59

### Grading
Each practice has ***totally 20 points***. You have to complete objectives to get points.<br>
<span style="color:red">***Primary objective***</span> : basic objective you must complete.<br>
<span style="color:orange">***Bonus objective***</span> : a little bit difficult objective for other bonus points.<br>
This practice has 5 <span style="color:red">***Primary objectives***</span> and 1 <span style="color:orange">***Bonus objective***</span>.


---
## Background: From the Pinhole Equation to a Linear System

A calibrated camera maps a 3D world point $\mathbf{X} = (X, Y, Z, 1)^\top$ to a 2D image point $\mathbf{x} = (u, v, 1)^\top$ through the **3×4 projection matrix** $P$:

$$\lambda\,\mathbf{x} = P\,\mathbf{X}$$

The unknown scalar $\lambda$ (the depth) is annoying. We can **eliminate** it by using the cross product: any vector is parallel to itself, so $\mathbf{x} \times (P\mathbf{X}) = \mathbf{0}$.

The cross product $\mathbf{x} \times \mathbf{y}$ can be written as a matrix multiplication using the **skew-symmetric (cross-product) matrix** $[\mathbf{x}]_\times$:

$$[\mathbf{x}]_\times \;=\; \begin{bmatrix} 0 & -1 & v \\ 1 & 0 & -u \\ -v & u & 0 \end{bmatrix} \quad\text{such that}\quad \mathbf{x} \times \mathbf{y} = [\mathbf{x}]_\times \mathbf{y}$$

So the projection constraint becomes a linear equation in $\mathbf{X}$:

$$[\mathbf{x}]_\times \, P \, \mathbf{X} \;=\; \mathbf{0}$$

This is one block of **3 equations** (only **2 are linearly independent** because $[\mathbf{x}]_\times$ is rank-2). With **N cameras** observing the same 3D point, we stack:

$$\underbrace{\begin{bmatrix} [\mathbf{x}_1]_\times P_1 \\ [\mathbf{x}_2]_\times P_2 \\ \vdots \\ [\mathbf{x}_N]_\times P_N \end{bmatrix}}_{A\;:\;(3N \times 4)} \mathbf{X} \;=\; \mathbf{0}$$

For **N = 3 cameras** the matrix $A$ is **9×4**, which is overdetermined for the 4 unknowns of $\mathbf{X}$. We solve $A\mathbf{X} = \mathbf{0}$ in the **least-squares** sense via SVD: the solution is the **right singular vector with the smallest singular value** (the last row of $V^\top$). Finally, we **dehomogenize** $\mathbf{X} = (X_1, X_2, X_3, X_4)^\top$ by dividing by $X_4$ to recover Cartesian $(X, Y, Z)$.

> **Practical note (beyond the slides):** Triangulation linearised this way is called **DLT triangulation**. It is fast and closed-form, and is the standard initialisation for non-linear refinement (e.g. minimising reprojection error). The DLT solution itself does *not* minimise reprojection error — it minimises an **algebraic** error that has no direct geometric meaning. For high-accuracy applications you would follow the DLT step with Levenberg–Marquardt refinement, but for clean correspondences (as in this practice) DLT alone is already accurate.


---
## Setup: Cameras and Correspondences

The scene contain a cube. And the 7 vertices in view marked as A~G, as shown below.

![](instruction_0.png)

We prepare 3 camera to capture the cube and reconstruct it. This 3 identical cameras are arrange horizontally and annotated with **left (l), middle (m), right (r)**. The cameras share the same intrinsics $K$ and orientation.

The three projection matrices are given as:

$$
P_l = \begin{bmatrix} 1440 & 0 & 500 & 3940 \\ 0 & 1440 & 500 & 2500 \\ 0 & 0 & 1 & 5 \end{bmatrix},\quad
P_m = \begin{bmatrix} 1440 & 0 & 500 & 2500 \\ 0 & 1440 & 500 & 2500 \\ 0 & 0 & 1 & 5 \end{bmatrix},\quad
P_r = \begin{bmatrix} 1440 & 0 & 500 & 1060 \\ 0 & 1440 & 500 & 2500 \\ 0 & 0 & 1 & 5 \end{bmatrix}
$$



And we have manually selected **7 corresponding image vertices** across the three views. The pixel coordinates are listed below, and the figure below shows where they sit in each view (A~G).

![](instruction_1.png)


Fill the value into ```p2ds_l```, ```p2ds_m```, ```p2ds_r``` as numpy arrays.

In [ ]:
import numpy as np

# Projection matrices of the left, mid, and right cameras (3 x 4)
P_l = np.array([[1.44e+03, 0.00e+00, 5.00e+02, 3.94e+03],
                [0.00e+00, 1.44e+03, 5.00e+02, 2.50e+03],
                [0.00e+00, 0.00e+00, 1.00e+00, 5.00e+00]], dtype=float)

P_m = np.array([[1.44e+03, 0.00e+00, 5.00e+02, 2.50e+03],
                [0.00e+00, 1.44e+03, 5.00e+02, 2.50e+03],
                [0.00e+00, 0.00e+00, 1.00e+00, 5.00e+00]], dtype=float)

P_r = np.array([[1.44e+03, 0.00e+00, 5.00e+02, 1.06e+03],
                [0.00e+00, 1.44e+03, 5.00e+02, 2.50e+03],
                [0.00e+00, 0.00e+00, 1.00e+00, 5.00e+00]], dtype=float)

# 7 corresponding 2D points in the three views (pixel coordinates)
p2ds_l = np.array([], dtype=float)
p2ds_m = np.array([], dtype=float)
p2ds_r = np.array([], dtype=float)

# Check the shape. all of them must be 7 by 2.
print('p2ds_l shape:', p2ds_l.shape)
print('p2ds_m shape:', p2ds_m.shape)
print('p2ds_r shape:', p2ds_r.shape)

---
## Part 1: The Cross-Product Matrix $[\mathbf{x}]_\times$

Given a 3-vector $\mathbf{x} = (x_1, x_2, x_3)^\top$, its skew-symmetric (cross-product) matrix is:

$$[\mathbf{x}]_\times \;=\; \begin{bmatrix} 0 & -x_3 & x_2 \\ x_3 & 0 & -x_1 \\ -x_2 & x_1 & 0 \end{bmatrix}$$

It satisfies $[\mathbf{x}]_\times \mathbf{y} = \mathbf{x} \times \mathbf{y}$ for any 3-vector $\mathbf{y}$.

For an image point in **homogeneous** coordinates $\mathbf{x} = (u, v, 1)^\top$:

$$[\mathbf{x}]_\times \;=\; \begin{bmatrix} 0 & -1 & v \\ 1 & 0 & -u \\ -v & u & 0 \end{bmatrix}$$


<span style="color:red">***Primary objective (1/5, 4 points):***</span> Implement the function `skew(v)` that returns the 3×3 cross-product matrix of a 3-vector `v`.

Then verify it on `v = [1, 2, 3]` and `w = [4, 5, 6]`:
- Print `skew(v)`
- Print `skew(v) @ w` and confirm it matches `np.cross(v, w)`


In [ ]:
def skew(v):
    """
    Return the 3x3 skew-symmetric (cross-product) matrix of a 3-vector v,
    such that skew(v) @ w == np.cross(v, w) for any 3-vector w.
    """
    ## YOUR IMPLEMENTATION ##

    M = None  # Your code here

    ## YOUR IMPLEMENTATION ##
    return M


# Verify
v = np.array([1.0, 2.0, 3.0])
w = np.array([4.0, 5.0, 6.0])
print('skew(v):')
print(skew(v))
print('skew(v) @ w  :', skew(v) @ w)
print('np.cross(v,w):', np.cross(v, w))

---
## Part 2: Build the Linear System for One 3D Point

For one 3D point $\mathbf{X}$ observed by the three cameras with image points $\mathbf{x}_l, \mathbf{x}_m, \mathbf{x}_r$ (in homogeneous coordinates), the constraints are:

$$[\mathbf{x}_l]_\times P_l \mathbf{X} = \mathbf{0},\quad [\mathbf{x}_m]_\times P_m \mathbf{X} = \mathbf{0},\quad [\mathbf{x}_r]_\times P_r \mathbf{X} = \mathbf{0}$$

Stacking these three blocks vertically gives the **9×4** matrix:

$$A \;=\; \begin{bmatrix} [\mathbf{x}_l]_\times P_l \\ [\mathbf{x}_m]_\times P_m \\ [\mathbf{x}_r]_\times P_r \end{bmatrix}, \qquad A\,\mathbf{X} = \mathbf{0}$$


<span style="color:red">***Primary objective (2/5, 4 points):***</span> Implement `build_A(x_l, x_m, x_r, P_l, P_m, P_r)` that:

1. Converts each 2D point to homogeneous form $(u, v, 1)$
2. Builds each 3×4 block $[\mathbf{x}_c]_\times P_c$
3. Vertically stacks the three blocks into a 9×4 matrix `A`

Test your function on the **first** correspondence (`p2ds_l[0]`, `p2ds_m[0]`, `p2ds_r[0]`) and print the resulting `A` and its shape.


In [ ]:
def build_A(x_l, x_m, x_r, P_l, P_m, P_r):
    """
    Build the 9x4 DLT system matrix A for one 3D point observed by 3 cameras.

    Args:
        x_l, x_m, x_r: each a (2,) array — pixel (u, v) in left/mid/right view
        P_l, P_m, P_r: each a (3, 4) projection matrix

    Returns:
        A: (9, 4) array such that A @ X = 0 for the homogeneous 3D point X
    """
    ## YOUR IMPLEMENTATION ##

    A = None  # Your code here

    ## YOUR IMPLEMENTATION ##
    return A


# Test on the first correspondence
A0 = build_A(p2ds_l[0], p2ds_m[0], p2ds_r[0], P_l, P_m, P_r)
print('A shape:', A0.shape)
print('A for point P1:')
print(A0)

---
## Part 3: Solve $A\mathbf{X} = \mathbf{0}$ via SVD

Compute the SVD $A = U \Sigma V^\top$. The least-squares solution to $A\mathbf{X} = \mathbf{0}$ subject to $\|\mathbf{X}\| = 1$ is the **right singular vector with the smallest singular value** — i.e. the **last row of $V^\top$** (equivalently, the last column of $V$).

The result $\hat{\mathbf{X}} = (X_1, X_2, X_3, X_4)^\top$ is in **homogeneous** coordinates. To recover the Cartesian 3D point we divide by the last component:

$$(X, Y, Z) \;=\; \bigl(X_1 / X_4,\; X_2 / X_4,\; X_3 / X_4\bigr)$$


<span style="color:red">***Primary objective (3/5, 4 points):***</span> Implement `triangulate_one(x_l, x_m, x_r, P_l, P_m, P_r)` that:

1. Calls `build_A` to get the 9×4 matrix
2. Computes the SVD and takes the last row of $V^\top$ as the homogeneous solution `X_h`
3. Dehomogenises and returns the Cartesian 3D point as a `(3,)` array

Test it on the first correspondence and print the recovered 3D point.


In [ ]:
def triangulate_one(x_l, x_m, x_r, P_l, P_m, P_r):
    """
    Triangulate one 3D point from its three image projections via DLT + SVD.

    Returns:
        X: (3,) Cartesian coordinates of the reconstructed 3D point
    """
    ## YOUR IMPLEMENTATION ##

    X = None  # Your code here

    ## YOUR IMPLEMENTATION ##
    return X


# Test on the first correspondence
X1 = triangulate_one(p2ds_l[0], p2ds_m[0], p2ds_r[0], P_l, P_m, P_r)
print('Reconstructed 3D point P1:', X1)

---
## Part 4: Triangulate All 7 Points

Once `triangulate_one` works, recovering every point in the dataset is a simple loop.


<span style="color:red">***Primary objective (4/5, 4 points):***</span> Triangulate all 7 correspondences and store the result in `p3ds` of shape `(7, 3)`.

Print the array. Sanity check: the reconstructed points should sit roughly **in front of** the cameras (positive $Z$, on the order of 1) and span roughly $\pm 0.5$ in $X$ and $Y$ — consistent with how they are laid out in each view.


In [ ]:
## YOUR IMPLEMENTATION ##

p3ds = None  # shape (7, 3) — reconstructed 3D points

## YOUR IMPLEMENTATION ##

print('p3ds shape:', p3ds.shape)
print('Reconstructed 3D points:')
for i, X in enumerate(p3ds):
    print(f'  P{i+1}: ({X[0]: .4f}, {X[1]: .4f}, {X[2]: .4f})')

---
## Part 5: Verify
There's the actual positions (ground truth) of 7 vertices:

![](instruction_2.png)


The **reconstruction error** is the Euclidean distance between the ground truth 3D point $\mathbf{X}$ and the reconstructed one $\hat{\mathbf{X}}$. 

$$e_c \;=\; \| \mathbf{X} - \hat{\mathbf{X}} \|_2$$

For perfect, noise-free correspondences, the error should be on the order of $10^{-6}$ or smaller (limited only by floating-point precision).

Since the corresponding points are selected by TA, we expected to get the un-perfect reconstruction result.

<span style="color:red">***Primary objective (5/5, 4 points):***</span> Compute the reconstruction error of every reconstructed point and print it out.

Also print the **mean** reconstruction error across all 7 points.


In [ ]:
## YOUR IMPLEMENTATION ##

errors = None  # must be (7, ) array  
errors_mean = None # mean of 7 points

## YOUR IMPLEMENTATION ##

for i in range(7):
    print(f'P{i+1:<5} {errors[i]:>14.6e}')

print(f'Mean reprojection error: {errors_mean:.6e}')

---
## Bonus Part: Two-View Triangulation Comparison

We can also reconstruct 3D point using only **two-view** triangulation. 
In this bonus, you will triangulate the same 7 points using only **two** of the three cameras (your choice — e.g. left & right) and compare the result with the previous 3-view DLT solution.


<span style="color:orange">***Bonus objective (4 points):***</span>

1. Pick **two** cameras (e.g. left and right) and 2 view triangulation at all 7 points. 
2. Dehomogenise and store as `p3ds_2view` of shape `(7, 3)`.
3. Compute the mean reconstruction error between `p3ds_2view` and ground truth from Part 5.
4. Print the mean error.
5. Compare the result of 3-view and 2-view DLT and print a one-line comment on what you observe.

In [ ]:
## YOUR IMPLEMENTATION ##
p3ds_2view = None
errors_mean_2view = None
## YOUR IMPLEMENTATION ##